In [0]:
Zona = "Palermo"
Precio = 150000
Moneda = "USD"
Metros= 27.3
Ambientes = 3
Tiene_Cochera = True

Precio_por_metro = round(Precio/Metros,2)

In [0]:
print(type(Zona))
print(type(Precio))
print(type(Moneda))
print(type(Metros))
print(type(Ambientes))
print(type(Tiene_Cochera))
print(Precio_por_metro)
print(type(Precio_por_metro))

In [0]:
Propiedades ={Zona : "Palermo",
Precio : 150000,
Moneda : "USD",
Metros: 27.3,
Ambientes : 3,
Tiene_Cochera : True}

print(f"La zona de la propiedad es: {Propiedades[Zona]}, con un valor de {Propiedades[Moneda]} {Propiedades[Precio]:,}")


In [0]:
# f-string: tabla alineada
Propiedades = [
    {"zona": "Palermo", "precio": 185000, "metros": 92.5},
    {"zona": "Belgrano", "precio": 320000, "metros": 145.0},
    {"zona": "Caballito", "precio": 78000, "metros": 55.0},
    {"zona": "Tigre", "precio": 250000, "metros": 200.0},
]

print(f"\n {'zona':<15} {'precio':>12} {'metros':>8} {'USD/m2':>10}")
print("-" * 47)
for prop in Propiedades:
    pm2 = round(prop["precio"] / prop["metros"], 2)
    print(f"{prop['zona']:<15} {prop['precio']:>12,} {prop['metros']:>8.3f} {pm2:>10.2f}")

In [0]:
Precios= [100000,50000,150000,750665,200000,800000]

print(len(Precios))
print(sum(Precios))
print(min(Precios))
print(max(Precios))
print(sorted(Precios))
print((Precios)[0])
print((Precios)[-1])
Precios.append(900000)
print(len(Precios))

In [0]:
zonas_todas = ["Palermo", "Belgrano", "Palermo", "Caballito", "Belgrano", "Tigre", "Palermo"]
zonas_unicas = set(zonas_todas)
print(len(zonas_todas))
print(len(zonas_unicas))

zonas_caba = {"Palermo", "Belgrano", "Caballito"}
zonas_gba = {"Tigre", "San Isidro", "Vicente López"}
print(f"\nUnión:        {zonas_caba | zonas_gba}")
print(f"Intersección: {zonas_caba & zonas_gba}")
print(f"Solo CABA:    {zonas_caba - zonas_gba}")

In [0]:

propiedad = {
    "zona": "Palermo",
    "precio": 185000,
    "moneda": "USD",
    "metros": 92.5,
    "ambientes": 3
}


print(f"Zona: {propiedad['zona']}")
print(f"Keys: {list(propiedad.keys())}")
print(f"Values: {list(propiedad.values())}")


propiedad["precio_por_m2"] = round(propiedad["precio"] / propiedad["metros"], 2)
print(f"\nCon precio_por_m2: {propiedad['precio_por_m2']} USD/m2")


print("\nContenido completo:")
for clave, valor in propiedad.items():
    print(f"  {clave}: {valor}")

In [0]:
# Lista de propiedades como dicts
propiedades = [
    {"zona": "Palermo", "precio": 185000, "metros": 92.5},
    {"zona": "Belgrano", "precio": 320000, "metros": 145.0},
    {"zona": "Caballito", "precio": 78000, "metros": 55.0},
    {"zona": "Terreno vacío", "precio": 45000, "metros": 0},
]

def precio_mt2(precio,metros):
    if metros== 0:
        return 0
    return round(precio/metros,2)

print(f"{'#':<4} {'Zona':<15} {'Precio':>10} {'Metros':>8} {'USD/m2':>10}")
print("-" * 49)
for i, prop in enumerate(propiedades, start=1):
    pm2 = precio_mt2(prop["precio"], prop["metros"])
    pm2_str = f"{pm2:,.2f}" if pm2 is not None else "N/A"
    print(f"{i:<4} {prop['zona']:<15} {prop['precio']:>10,} {prop['metros']:>8.1f} {pm2_str:>10}")

# range() para repetir N veces
print("\nConteo regresivo:")
for i in range(3, 0, -1):
    print(f"  {i}...")




In [0]:
# Loop sobre zonas con spark.sql() + f-string
zonas = ["Capital Federal", "Vicente López", "San Isidro", "Tigre"]

print(f"{'Zona':<20} {'Propiedades':>12}")
print("-" * 34)

for zona in zonas:
    n = spark.sql(f"""
        select count(*) as cantidad_props
        from bootcamp.silver.propiedades
        where partido ='{zona}'
    """).collect()[0]["cantidad_props"]
    print(f"{zona:<20} {n:>12,}")
    

In [0]:

tabla = "bootcamp.gold.dim_zona"
print(f"Estructura de {tabla}:")
spark.sql(f"DESCRIBE {tabla}").show(truncate=False)

In [0]:
# SparkSession — ya existe en Databricks
print(f"Spark version: {spark.version}")b
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

In [0]:
from pyspark.sql.functions import col

df = spark.table("bootcamp.silver.propiedades")

df_filtrado = (
    df
    .filter(col("tipo_operacion") == "Venta")
    .filter(col("moneda") == "USD")
    .select("partido", "precio", "metros_cuadrados_totales", "precio_por_m2")
)


In [0]:
# .explain(True) — ver plan lógico y físico
df_filtrado.explain(True)

In [0]:
from pyspark.sql.functions import col, avg, count

# Forma 1: DataFrame API
resultado_df = (
    spark.table("bootcamp.silver.propiedades")
    .filter(col("tipo_operacion") == "Venta")
    .groupBy("partido")
    .agg(
        count("*").alias("cantidad"),
        avg("precio").alias("precio_promedio")
    )
)
resultado_df.show(5)

In [0]:
resultado_sql = spark.sql("""
    SELECT partido, COUNT(*) as cantidad, AVG(precio) as precio_promedio
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion = 'Venta'
    GROUP BY partido
""")
resultado_sql.show(5)

In [0]:
%sql

SELECT partido, COUNT(*) as cantidad, AVG(precio) as precio_promedio
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'venta'
GROUP BY partido
LIMIT 5;

In [0]:
tabla=spark.table("bootcamp.silver.propiedades")
tabla.printSchema()
print(f"{tabla.count()}")

In [0]:
from pyspark.sql.functions import col
tabla.select("partido","ambientes","antiguedad").show(5)

tabla.select(
    col("partido"),
    col("moneda"),
    col("precio"),
    (col("precio")/1000).alias("precio_miles")
).show(5)

tabla_sin_url= tabla.drop("url","fecha_publicacion")
print(f"original{len(tabla.columns)}, nueva {len(tabla_sin_url.columns)}")

tabla.withColumnRenamed("estado","Estado Nueva").select("partido","ambientes","Estado Nueva").show(5)


In [0]:
tabla_reduc= tabla.drop("url","fecha_publicacion","_source_table","region")

tabla_reduc.filter(col("tipo_operacion")=="venta").select("partido","tipo_operacion","moneda","precio").show(5)

tabla_reduc.filter(col("precio").between (100000,200000)).select("partido","tipo_operacion","moneda","precio").show(5)

tabla_reduc.filter(col("partido").contains ("federal")).select("partido","tipo_operacion","moneda","precio").show(5)

tabla_reduc.filter(col("partido").isin("capital federal","vicente lopez","tigre","moron")).select("partido","tipo_operacion","moneda","precio").show(5)

tabla_reduc.filter(~col("moneda").isin("USD")).select("partido", "moneda", "precio").show(5)

In [0]:
Tabla_filtrada= spark.table("bootcamp.silver.propiedades").filter(col("tipo_operacion")=="venta").filter(col("moneda")=="USD").filter(col("metros_cuadrados_totales")>100).select("partido","precio","metros_cuadrados_totales","precio_por_m2","ambientes")

In [0]:
Tabla_filtrada.explain(True)

In [0]:
Tabla_filtrada.show(5)

In [0]:
from pyspark.sql.functions import when

# withColumn + when = CASE WHEN de SQL
df_categorizado = Tabla_filtrada.withColumn(
    "categoria_precio",
    when(col("precio") < 80000, "Económica")
    .when(col("precio") < 200000, "Media")
    .when(col("precio") < 500000, "Premium")
    .otherwise("Lujo")
)
df_categorizado.show(15)

In [0]:
df_silver= spark.table("bootcamp.silver.propiedades").filter(col("moneda")=="ARS").filter(col("ambientes")>0).select(
    col("partido"),col("precio"),col("ambientes"),col("metros_cuadrados_totales"),(col("precio")/col("ambientes")).alias("precio_por_ambiente")).orderBy(col("precio_por_ambiente").desc())
df_silver.show(20)



In [0]:
# Misma query en spark.sql()
resultado_sql = spark.sql("""
    SELECT 
        partido,
        precio,
        ambientes,
        metros_cuadrados_totales,
        CAST(precio / ambientes AS DECIMAL(10,2)) AS precio_por_ambiente
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion = 'alquiler' 
      AND moneda = 'ARS'
      AND ambientes > 0
    ORDER BY precio_por_ambiente DESC
""")
resultado_sql.show(20)

In [0]:
print("=== DataFrame API ===")
df_silver.explain()

print("\n=== spark.sql() ===")
resultado_sql.explain()

In [0]:
zona= ["capital federal","tigre","moron","avellaneda"]

for z in zona:
    resultado = spark.sql(f"""
        select count(*) as cantidad from bootcamp.silver.propiedades where partido='{z}' """).collect()[0]
    
    print(f'la zona {z}, tiene {resultado['cantidad']} avisos')